# DuckDB and Polars on OneLake Iceberg

Read and write Iceberg tables in a Fabric lakehouse through the OneLake Iceberg REST catalog,
from DuckDB 2 and Polars 2.1. No secrets: the token comes from `notebookutils`.

The lakehouse must have **Lakehouse schemas** enabled.

## Install
The runtime ships older versions, so upgrade and restart Python.

In [ ]:
%pip install -q --upgrade "duckdb>=2" "polars[iceberg]>=2.1" "pyiceberg[pyarrow,adlfs]>=0.12"
notebookutils.session.restartPython()

## Parameters
Leave empty to use this notebook's workspace and default lakehouse, or paste the GUIDs (names don't resolve).

In [ ]:
workspace_id = ""
lakehouse_id = ""

In [ ]:
ctx = notebookutils.runtime.context
workspace_id = workspace_id or ctx["currentWorkspaceId"]
lakehouse_id = lakehouse_id or ctx["defaultLakehouseId"]

warehouse = f"{workspace_id}/{lakehouse_id}"
token = notebookutils.credentials.getToken("storage")  # ~1 hour; rerun this cell to refresh
SCHEMA = "demo"
SF = 1

## DuckDB: generate TPC-H and write it to OneLake

In [ ]:
import duckdb

con = duckdb.connect()
con.sql(f"CALL dbgen(sf={SF})")  # in memory

con.sql(f"""
    CREATE OR REPLACE SECRET onelake_storage (
        TYPE azure, PROVIDER access_token, ACCESS_TOKEN '{token}');
    ATTACH OR REPLACE '{warehouse}' AS onelake (
        TYPE ICEBERG,
        URI 'https://onelake.table.fabric.microsoft.com/iceberg',
        TOKEN '{token}',
        ACCESS_DELEGATION_MODE 'none',             -- use the secret above, no vended credentials
        STAGE_CREATE_TABLES false,                 -- OneLake rejects staged creates
        SKIP_CREATE_TABLE_METADATA_UPDATES true);  -- and the follow-up commit after CREATE
""")

con.sql(f"CREATE SCHEMA IF NOT EXISTS onelake.{SCHEMA}")
for table in ["lineitem", "orders"]:
    con.sql(f"DROP TABLE IF EXISTS onelake.{SCHEMA}.{table}")
    con.sql(f"CREATE TABLE onelake.{SCHEMA}.{table} AS SELECT * FROM memory.{table}")
    print(table, con.sql(f"SELECT count(*) FROM onelake.{SCHEMA}.{table}").fetchall()[0][0], "rows")

## DuckDB: read

In [ ]:
q1_duckdb = con.sql(f"""
    SELECT l_returnflag, l_linestatus,
           sum(l_quantity) AS sum_qty,
           sum(l_extendedprice) AS sum_base_price,
           sum(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
           count(*) AS count_order
    FROM onelake.{SCHEMA}.lineitem
    WHERE l_shipdate <= DATE '1998-09-02'
    GROUP BY ALL
    ORDER BY ALL
""")
q1_duckdb.show()

## Polars: attach the catalog

In [ ]:
import polars as pl
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "onelake",
    uri="https://onelake.table.fabric.microsoft.com/iceberg",
    warehouse=warehouse,
    token=token,  # signs the REST calls
    **{
        "adls.account-name": "onelake",
        "adls.account-host": "onelake.blob.fabric.microsoft.com",
        "adls.token": token,  # reads and writes the files
    },
)
storage_options = {"bearer_token": token}
catalog.list_tables(SCHEMA)

## Polars: read

In [ ]:
lineitem = pl.scan_iceberg(f"{SCHEMA}.lineitem", catalog=catalog, storage_options=storage_options)

q1_polars = (
    lineitem.filter(pl.col("l_shipdate") <= pl.date(1998, 9, 2))
    .group_by("l_returnflag", "l_linestatus")
    .agg(
        sum_qty=pl.col("l_quantity").sum(),
        sum_base_price=pl.col("l_extendedprice").sum(),
        sum_disc_price=(pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).sum(),
        count_order=pl.len().cast(pl.Int64),
    )
    .sort("l_returnflag", "l_linestatus")
    .collect()
)
q1_polars

SQL works too, once the catalog is registered.

In [ ]:
sql = pl.SQLContext().register_catalog("onelake", catalog, storage_options=storage_options)
sql.execute(f"""
    SELECT o_orderpriority, count(*) AS orders
    FROM onelake.{SCHEMA}.orders
    GROUP BY o_orderpriority
    ORDER BY o_orderpriority
""", eager=True)

## Polars: write

In [ ]:
q1_polars.lazy().sink_iceberg(
    f"{SCHEMA}.q1_polars",
    catalog=catalog,
    mode="overwrite",
    create_if_not_exists=True,
    storage_options=storage_options,
)

## DuckDB reads what Polars wrote

In [ ]:
from_polars = con.sql(f"SELECT * FROM onelake.{SCHEMA}.q1_polars ORDER BY ALL")
from_polars.show()
a, b = from_polars.fetchall(), q1_duckdb.fetchall()
assert a == b, f"DuckDB and Polars disagree on Q1: {from_polars.types} {a} vs {q1_duckdb.types} {b}"
print("DuckDB and Polars agree")